# Lección 10.2 — Convoluciones: filtros que detectan patrones

> **Módulo 10 · Visión por computadora (CNN)** · Nivel 4 — Avanzado
> ⏱️ Tiempo estimado: 100–120 min (lección con ejercicios intercalados ≈ 80 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 10.1 (la imagen como tabla de números), Lección 9.4 (retropropagación), Lección 9.5 (optimizadores)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Calcular **a mano** una convolución: multiplicar un filtro por un trozo de la imagen y sumar.
- Reconocer qué detectan algunos filtros clásicos (bordes verticales, horizontales, desenfoque, nitidez).
- Predecir el tamaño de salida según el **tamaño de kernel**, el **stride** y el **padding**, y contar los parámetros de una capa.
- Usar `nn.Conv2d` de PyTorch y ver cómo una red **aprende sola** un filtro.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.
>
> 🖥️ Todo corre en CPU en pocos minutos (también en Google Colab sin GPU). Si tienes GPU, se usará sola.

## ▶️ Preparación

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.datasets import load_sample_image

plt.rcParams["figure.figsize"] = (8, 4.5)
torch.manual_seed(0)
foto_gris = load_sample_image("china.jpg").mean(axis=2)[::2, ::2] / 255   # foto en grises, a la mitad de tamaño, 0–1
print("Foto en grises:", foto_gris.shape)
print("✅ Todo listo")

---
## 1. 📖 La intuición: una linterna que busca un patrón

Imagina que buscas a Wally en un libro gigante con una **lupa pequeña**. No miras toda la página a la vez: recorres la página **trozo a trozo**, y en cada trozo te preguntas lo mismo: *"¿veo aquí un gorro de rayas rojas y blancas?"*.

Una **convolución** hace eso:
- La **lupa** es un **filtro** (o *kernel*): una tablita pequeña de números, por ejemplo de 3 × 3.
- En cada posición, el filtro mira el trozo de imagen que tiene debajo y responde con **un número**: *"¿cuánto se parece este trozo a mi patrón?"*.
- Después **se desliza** un píxel y repite. Al final tienes un **mapa** de dónde aparece el patrón: el **mapa de características** (*feature map*).

Y lo más importante: **es la misma lupa en toda la página**. Si Wally se mueve, la lupa lo sigue encontrando. Eso es justo lo que le faltaba al modelo "píxel por píxel" de la Lección 10.1.

### ✋ Ejercicio 1 — Calentamiento
En la analogía de la lupa, ¿qué ventaja tiene usar **la misma** lupa en toda la página en vez de una lupa distinta para cada zona? Menciona dos.

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Funciona en cualquier lugar**: si el patrón se mueve, la misma lupa lo detecta en su nueva posición.
2. **Necesita muy pocos números**: una lupa de 3 × 3 tiene 9 números, sin importar si la imagen es de 28 × 28 o de 4,000 × 3,000. A esto se le llama **compartir pesos** (*weight sharing*).

(Una tercera: lo que la lupa aprende en una zona de la imagen sirve automáticamente para todas las demás.)
</details>

---
## 2. 📖 El concepto: multiplicar y sumar

En cada posición, la convolución hace una cuenta muy simple: **multiplica cada número del filtro por el píxel que tiene debajo y suma todo**.

```
 Trozo de imagen        Filtro            Producto casilla a casilla
 ┌───┬───┬───┐       ┌───┬───┬───┐       ┌───┬───┬───┐
 │ 0 │ 0 │ 1 │       │-1 │ 0 │ 1 │       │ 0 │ 0 │ 1 │
 ├───┼───┼───┤   ×   ├───┼───┼───┤   =   ├───┼───┼───┤   → suma = 4
 │ 0 │ 0 │ 1 │       │-2 │ 0 │ 2 │       │ 0 │ 0 │ 2 │
 ├───┼───┼───┤       ├───┼───┼───┤       ├───┼───┼───┤
 │ 0 │ 0 │ 1 │       │-1 │ 0 │ 1 │       │ 0 │ 0 │ 1 │
 └───┴───┴───┘       └───┴───┴───┘       └───┴───┴───┘
```

Este filtro (llamado **Sobel**) resta lo que hay a la izquierda y suma lo que hay a la derecha: da un número **grande** cuando la derecha es más clara que la izquierda, o sea, cuando hay un **borde vertical**. Si el trozo es parejo (todo igual), da **0**.

### ✋ Ejercicio 2 — Calcula a mano
Con el mismo filtro Sobel, calcula el resultado para estos dos trozos:

```
Trozo A           Trozo B
5 5 5             9 0 0
5 5 5             9 0 0
5 5 5             9 0 0
```
✍️ *Mis resultados:* A → ___ · B → ___

Comprueba con la celda de abajo.

In [ ]:
sobel_x = np.array([[-1, 0, 1],
                    [-2, 0, 2],
                    [-1, 0, 1]])

trozo_A = np.full((3, 3), 5)
trozo_B = np.array([[9, 0, 0], [9, 0, 0], [9, 0, 0]])

print("Trozo A →", (trozo_A * sobel_x).sum())
print("Trozo B →", (trozo_B * sobel_x).sum())

<details><summary><b>👀 Ver respuesta</b></summary>

- **A → 0**: un trozo parejo no tiene bordes; lo de la izquierda y lo de la derecha se cancelan.
- **B → −36**: hay un borde, pero al revés (claro a la izquierda, oscuro a la derecha): −9 − 18 − 9 = −36. El **signo** dice la dirección del borde y el **tamaño** dice qué tan fuerte es.
</details>

### 📖 Deslizar el filtro por toda la imagen
Ahora repetimos esa cuenta en **cada posición**. Esta es una convolución completa, escrita a mano con NumPy (son solo dos bucles):

In [ ]:
def convolucion(imagen, filtro, stride=1, padding=0):
    """Convolución 2D a mano: desliza el filtro, multiplica y suma."""
    imagen = np.pad(imagen, padding)                 # padding: marco de ceros alrededor
    kh, kw = filtro.shape
    alto = (imagen.shape[0] - kh) // stride + 1
    ancho = (imagen.shape[1] - kw) // stride + 1
    salida = np.zeros((alto, ancho))
    for i in range(alto):
        for j in range(ancho):
            trozo = imagen[i * stride: i * stride + kh, j * stride: j * stride + kw]
            salida[i, j] = (trozo * filtro).sum()
    return salida

# Imagen de 6×6: mitad izquierda negra (0), mitad derecha blanca (1)
mitades = np.zeros((6, 6)); mitades[:, 3:] = 1
print(mitades)

### 🔮 Ejercicio 3 — ¿Dónde se enciende?
Vamos a pasar el filtro `sobel_x` por la imagen `mitades` (sin padding, stride 1).
1. ¿De qué tamaño será la salida?
2. ¿Dónde aparecerán los números grandes y dónde los ceros?

✍️ *Mis predicciones:*

In [ ]:
print(convolucion(mitades, sobel_x))

<details><summary><b>👀 Ver respuesta</b></summary>

1. **4 × 4**: un filtro de 3 × 3 solo cabe en 4 posiciones por fila en una imagen de 6 de ancho (6 − 3 + 1 = 4).
2. Las dos columnas del **centro** valen **4** (ahí el filtro está montado sobre la frontera negro/blanco) y las de los lados valen **0** (zona pareja). El mapa de salida dice **dónde** está el borde.
</details>

### ✋ Ejercicio 4 — Gira el filtro
Si **trasponemos** el filtro (`sobel_x.T`, filas ↔ columnas), detecta bordes **horizontales**. ¿Qué dará sobre `mitades`? ✍️ *Mi apuesta:*

In [ ]:
sobel_y = sobel_x.T
print(sobel_y)
print(convolucion(mitades, sobel_y))

<details><summary><b>👀 Ver respuesta</b></summary>

**Todo ceros.** La imagen `mitades` solo tiene un borde **vertical**; de arriba abajo no cambia nada, así que el detector de bordes horizontales no encuentra nada. Cada filtro **busca un patrón concreto** y es "ciego" a lo demás. Por eso una red usa **muchos** filtros a la vez.
</details>

---
## 3. 📖 Cómo funciona con imágenes reales: una galería de filtros

Probemos cuatro filtros clásicos sobre la foto del templo. Los cuatro tienen 3 × 3 números; solo cambia **cuáles**:

| Filtro | Números | Qué hace |
|---|---|---|
| Bordes verticales | `sobel_x` | Resta izquierda, suma derecha |
| Bordes horizontales | `sobel_y` | Resta arriba, suma abajo |
| Desenfoque | todos 1/9 | Promedia los vecinos |
| Nitidez | 5 en el centro, −1 en los vecinos | Exagera la diferencia con los vecinos |

### 🔮 Ejercicio 5 — Empareja
La celda de abajo muestra los resultados **desordenados** (A, B, C, D). Antes de leer la respuesta, decide qué filtro produjo cada uno. Pista: el templo tiene **columnas verticales** y **tejados horizontales**.

In [ ]:
filtros = {
    "bordes verticales": sobel_x,
    "bordes horizontales": sobel_y,
    "desenfoque": np.ones((3, 3)) / 9,
    "nitidez": np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]]),
}
orden = ["bordes horizontales", "desenfoque", "bordes verticales", "nitidez"]   # desordenados a propósito

recorte = foto_gris[40:160, 60:220]      # un trozo del templo, para verlo de cerca
fig, ax = plt.subplots(1, 5, figsize=(18, 3.2))
ax[0].imshow(recorte, cmap="gray"); ax[0].set_title("Original")
for a, letra, nombre in zip(ax[1:], "ABCD", orden):
    salida = convolucion(recorte, filtros[nombre])
    a.imshow(np.abs(salida) if "bordes" in nombre else salida, cmap="gray"); a.set_title(letra)
for a in ax: a.axis("off")
plt.show()

✍️ *Mis parejas:* A → ___ · B → ___ · C → ___ · D → ___

<details><summary><b>👀 Ver respuesta</b></summary>

**A → bordes horizontales** (resaltan los tejados y los escalones) · **B → desenfoque** (la imagen se ve borrosa) · **C → bordes verticales** (resaltan columnas y marcos) · **D → nitidez** (se parece al original pero con contornos más marcados).

(En los detectores de bordes dibujamos el **valor absoluto**, para que se vean igual los bordes de oscuro a claro y de claro a oscuro.)

Durante décadas, los expertos en visión **diseñaban estos filtros a mano**. La gran idea de las redes convolucionales es que **la red aprende sola qué filtros necesita** (Sección 3.2).
</details>

### 📖 3.1 Lo mismo en PyTorch: `nn.Conv2d`
PyTorch tiene la convolución lista y muy optimizada. Comprobemos que hace **exactamente** la misma cuenta que nuestra función: le ponemos a mano el filtro Sobel como peso y comparamos.

> 📝 Técnicamente, lo que hacen PyTorch y nuestra función se llama **correlación cruzada** (la convolución "de los libros de matemáticas" voltea el filtro antes). En deep learning todo el mundo lo llama convolución y da igual, porque los filtros se aprenden.

In [ ]:
capa = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3, bias=False)
with torch.no_grad():
    capa.weight[:] = torch.tensor(sobel_x, dtype=torch.float32)   # le "instalamos" el filtro Sobel

entrada = torch.tensor(foto_gris, dtype=torch.float32)[None, None]   # forma (1, 1, H, W): lote de 1, 1 canal
salida_torch = capa(entrada)[0, 0].detach().numpy()
salida_numpy = convolucion(foto_gris, sobel_x)

print("Forma NumPy  :", salida_numpy.shape)
print("Forma PyTorch:", salida_torch.shape)
print(f"Diferencia máxima entre ambas: {np.abs(salida_torch - salida_numpy).max():.7f}")

### ✋ Ejercicio 6
¿Por qué tuvimos que escribir `[None, None]` para crear la entrada? (Pista: Lección 10.1, Sección 3.)

<details><summary><b>👀 Ver respuesta</b></summary>

Porque `nn.Conv2d` espera un **lote** con forma `(N, C, H, W)`. La foto era solo `(H, W)`; cada `None` añade una dimensión: una para el **lote** (N = 1) y otra para los **canales** (C = 1, porque está en grises). La diferencia entre NumPy y PyTorch es prácticamente 0 (solo errores de redondeo): es la misma operación.
</details>

### 📖 3.2 La red aprende sus propios filtros
En una red convolucional **no** escribimos los números del filtro: empiezan **al azar** y se ajustan con descenso de gradiente y retropropagación (Lecciones 9.4 y 9.5), igual que los pesos de cualquier red.

Hagamos un experimento: le mostramos a una capa `Conv2d` con pesos aleatorios la foto y el resultado de aplicarle Sobel, **sin decirle qué filtro usamos**. ¿Lo descubrirá?

### 🔮 Ejercicio 7 — ¿Puede adivinar el filtro?
Antes de ejecutar: después de 300 pasos de entrenamiento, ¿crees que el filtro aprendido se parecerá a `sobel_x`?

In [ ]:
torch.manual_seed(0)
objetivo = torch.tensor(salida_numpy, dtype=torch.float32)[None, None]   # lo que queremos que produzca

aprendiz = nn.Conv2d(1, 1, kernel_size=3, bias=False)                   # pesos al azar
print("Filtro al inicio (azar):\n", aprendiz.weight.data[0, 0].numpy().round(2))

optimizador = torch.optim.Adam(aprendiz.parameters(), lr=0.05)
for paso in range(301):                                                 # ✏️ prueba con menos pasos (p. ej. 50)
    perdida = ((aprendiz(entrada) - objetivo) ** 2).mean()               # error cuadrático medio (Lección 3.2)
    optimizador.zero_grad(); perdida.backward(); optimizador.step()
    if paso % 100 == 0:
        print(f"paso {paso:>3} · pérdida {perdida.item():.6f}")

print("\nFiltro aprendido:\n", aprendiz.weight.data[0, 0].numpy().round(2))

<details><summary><b>👀 Ver respuesta</b></summary>

¡Sí! La pérdida baja de ≈0.17 a casi 0 y el filtro aprendido es, redondeado, **exactamente** `[[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]`. Con 50 pasos todavía está a medio camino.

En una red real nadie le dice a la red el "filtro correcto": solo le dice si acertó la **clase** (gato, perro…). Aun así, al entrenar, los filtros de la primera capa terminan pareciéndose a detectores de bordes y de colores, porque son útiles para la tarea. Lo verás en la Lección 10.3.
</details>

---
## 4. 📖 ¿Qué pasa si…? Imágenes a color y muchos filtros

Hasta ahora: 1 canal de entrada y 1 filtro. En una red real:
- **La entrada tiene varios canales** (3 en una foto RGB). Entonces cada filtro es un "cubo" de `3 × 3 × 3`: mira los tres canales a la vez y suma todo en **un solo** número por posición.
- **Hay muchos filtros** (16, 32, 64…). Cada uno produce su propio mapa de características. La salida de la capa tiene **tantos canales como filtros**.

```
Entrada (3, 32, 32)  ──16 filtros de 3×3×3──►  Salida (16, 32, 32)
   foto RGB                                       16 mapas: "¿dónde hay borde vertical?",
                                                  "¿dónde hay rojo?", "¿dónde hay esquina?"…
```

**Parámetros de una capa convolucional** = `filtros × (canales_entrada × k × k) + filtros` (el último sumando es un sesgo por filtro).

### 🔮 Ejercicio 8 — Cuenta los parámetros
Una capa `nn.Conv2d(3, 16, kernel_size=3)` recibe fotos de 32 × 32.
1. ¿Cuántos parámetros tiene?
2. ¿Cambiaría ese número si las fotos fueran de 1,000 × 1,000?
3. Compáralo con una capa **densa** que reciba la foto de 32 × 32 × 3 aplanada y tenga 16 neuronas.

✍️ *Mis cuentas:*

In [ ]:
conv = nn.Conv2d(3, 16, kernel_size=3)
densa = nn.Linear(32 * 32 * 3, 16)

contar = lambda capa: sum(p.numel() for p in capa.parameters())
print(f"Convolucional: {contar(conv):,} parámetros")
print(f"Densa        : {contar(densa):,} parámetros")
print("Salida de la conv con una foto de 32×32:", tuple(conv(torch.zeros(1, 3, 32, 32)).shape))

<details><summary><b>👀 Ver respuesta</b></summary>

1. 16 × (3 × 3 × 3) + 16 = **448**.
2. **No.** Los filtros son los mismos sin importar el tamaño de la imagen; solo cambia el tamaño del mapa de salida.
3. La densa necesita 3,072 × 16 + 16 = **49,168** parámetros, más de 100 veces más, y además **no** sabe que los píxeles vecinos están relacionados. Con fotos de 1,000 × 1,000 la densa tendría 48 millones.
</details>

---
## 5. 📖 Hiperparámetros: las perillas de una capa convolucional

| Hiperparámetro | Qué controla | Si lo **subes** ⬆️ | Si lo **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| `kernel_size` (tamaño del filtro) | Cuántos vecinos mira cada filtro | Ve patrones más grandes, pero con más parámetros y más cómputo | Patrones más locales y finos; más barato | **3** (casi siempre); 5–7 en la primera capa de algunas redes |
| `stride` (paso) | Cuántos píxeles salta el filtro entre una posición y la siguiente | Salida más pequeña y más rápida, pero pierde detalle | Salida más grande y detallada, más cómputo | **1**; 2 para reducir a la mitad |
| `padding` (relleno) | Marco de ceros alrededor de la imagen | Conserva el tamaño y la información de los bordes | La salida se encoge en cada capa | `kernel_size // 2` (p. ej. 1 con kernel 3: tamaño igual, *"same"*) |
| `out_channels` (número de filtros) | Cuántos patrones distintos busca la capa | Más patrones, más capacidad… y más riesgo de sobreajuste y más lento | Menos capacidad (subajuste) | 16–64 en las primeras capas; se suele **duplicar** al reducir el tamaño |

### 📏 La fórmula del tamaño de salida
$$\text{salida} = \frac{\text{entrada} + 2 \cdot \text{padding} - \text{kernel}}{\text{stride}} + 1 \quad\text{(redondeando hacia abajo)}$$

En palabras: *"cuántas veces cabe el filtro, dando saltos de tamaño `stride`, en la imagen con su marco"*.

### ✋ Ejercicio 9 — Usa la fórmula
Entrada de 32 × 32. Calcula el tamaño de salida para:

| | kernel | stride | padding | Tu respuesta |
|---|---|---|---|---|
| a) | 3 | 1 | 0 | |
| b) | 3 | 1 | 1 | |
| c) | 5 | 1 | 0 | |
| d) | 3 | 2 | 1 | |
| e) | 7 | 2 | 3 | |

Luego comprueba con PyTorch (también imprime cuántos parámetros tiene cada capa con 16 filtros):

In [ ]:
for k, s, p in [(3, 1, 0), (3, 1, 1), (5, 1, 0), (3, 2, 1), (7, 2, 3)]:   # ✏️ añade tus combinaciones
    capa = nn.Conv2d(3, 16, kernel_size=k, stride=s, padding=p)
    forma = tuple(capa(torch.zeros(1, 3, 32, 32)).shape)
    print(f"kernel={k} stride={s} padding={p} → salida {forma[2]}×{forma[3]}  ·  {contar(capa):>5,} parámetros")

<details><summary><b>👀 Ver respuesta</b></summary>

a) (32 − 3)/1 + 1 = **30** · b) (32 + 2 − 3)/1 + 1 = **32** (padding 1 con kernel 3 conserva el tamaño) · c) **28** · d) (32 + 2 − 3)/2 + 1 = 16.5 → **16** · e) (32 + 6 − 7)/2 + 1 = 16.5 → **16**.

Fíjate en los parámetros: el kernel de 7 × 7 cuesta **5 veces** más (2,368) que el de 3 × 3 (448). El *stride* y el *padding* no cambian el número de parámetros, solo el tamaño de la salida.
</details>

### 🔮 Ejercicio 10 — El stride en acción
Aplicamos el detector de bordes verticales a la foto con `stride` = 1, 2 y 4. ¿Cómo cambiarán las imágenes de salida? ✍️ *Mi predicción:*

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
for a, s in zip(ax, [1, 2, 4]):                    # ✏️ prueba stride 8
    salida = convolucion(foto_gris, sobel_x, stride=s)
    a.imshow(np.abs(salida), cmap="gray"); a.set_title(f"stride={s} → {salida.shape[0]}×{salida.shape[1]}"); a.axis("off")
plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

Con `stride=2` la salida tiene la **mitad** de alto y de ancho (la cuarta parte de píxeles), y con `stride=4` la cuarta parte de alto y de ancho. Las imágenes se ven cada vez más **pixeladas**: el filtro "salta" posiciones y se pierde detalle fino, a cambio de mucho menos cómputo en las capas siguientes. Reducir el tamaño de esta forma es una alternativa al *pooling* (Lección 10.3).
</details>

### ✏️ Ejercicio 11 — El padding y los bordes
Sin padding, cada capa con kernel 3 "come" un píxel de cada lado. Apila 10 capas así sobre una imagen de 28 × 28 (como MNIST). ¿Qué tamaño queda? ¿Y si usas `padding=1`? Cambia `PADDING` y comprueba.

In [ ]:
PADDING = 0   # ✏️ cambia a 1
x = torch.zeros(1, 1, 28, 28)
for i in range(10):
    x = nn.Conv2d(1, 1, kernel_size=3, padding=PADDING)(x)
    print(f"después de la capa {i + 1:>2}: {x.shape[2]}×{x.shape[3]}")

<details><summary><b>👀 Ver respuesta</b></summary>

Sin padding: 28 → 26 → 24 → … → **8 × 8** tras 10 capas (pierde 2 por capa). Con 14 capas ya no quedaría nada. Además, los píxeles de los bordes participan en menos cuentas: la red "ve peor" los bordes de la imagen.

Con `padding=1` el tamaño se mantiene en **28 × 28** en todas las capas. Por eso casi todas las redes modernas usan padding *"same"* (`kernel_size // 2`) y reducen el tamaño **a propósito** con *stride* o *pooling*, no por accidente.
</details>

---
## 6. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| *RuntimeError: Expected 4D input* | Pasaste una imagen sin dimensión de lote | `img.unsqueeze(0)` |
| *expected input to have 3 channels, but got 1* | `in_channels` no coincide con los canales de la entrada | Revisa la forma `(N, C, H, W)` y el `in_channels` de la primera capa |
| La imagen se encoge hasta desaparecer | Muchas capas sin padding o *stride* grande | `padding = kernel_size // 2` y reduce el tamaño solo donde quieras |
| Error de tamaño al pasar de la parte convolucional a la densa | Calculaste mal el tamaño de salida | Usa la fórmula o pasa un tensor de ceros y mira la forma (Lección 10.3) |
| Red lenta y con sobreajuste | Kernels grandes y demasiados filtros | Kernels de 3 × 3; menos filtros; regularización (Lección 9.8) |

### 🔮 Ejercicio 12 — Encuentra el error
Este código da error. ¿Por qué? Intenta arreglarlo (cambia **un solo número**).

In [ ]:
fotos = torch.rand(8, 3, 64, 64)                  # 8 fotos a color
capa_mal = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3)   # ✏️ arréglame
try:
    print(tuple(capa_mal(fotos).shape))
except RuntimeError as e:
    print("❌ Error:", str(e)[:120])

<details><summary><b>👀 Ver respuesta</b></summary>

Las fotos tienen **3 canales** (RGB), pero la capa espera **1**. Cambia `in_channels=1` por `in_channels=3` y la salida será `(8, 16, 62, 62)`: 8 fotos, 16 mapas de características de 62 × 62 (sin padding, se pierden 2 píxeles).
</details>

---
## 7. 📋 Ficha: la capa convolucional

| | Capa convolucional (`nn.Conv2d`) |
|---|---|
| **Intuición** | Una lupa (filtro) que recorre la imagen preguntando "¿veo aquí mi patrón?" |
| **Qué calcula** | En cada posición: multiplica el filtro por el trozo de imagen y suma → un mapa de características por filtro |
| **Cómo aprende** | Los números de los filtros son pesos: empiezan al azar y se ajustan con retropropagación |
| **Hiperparámetros clave** | `kernel_size`, `stride`, `padding`, `out_channels` |
| **✅ Fortalezas** | Pocos parámetros (pesos compartidos) · Detecta el patrón esté donde esté · Aprovecha la estructura de vecindario |
| **❌ Debilidades** | Cada filtro solo ve un vecindario pequeño: hace falta apilar capas para ver objetos grandes (10.3) · No es invariante a rotaciones ni a cambios de escala por sí sola |
| **Úsala cuando…** | Los datos tienen estructura de cuadrícula: imágenes, espectrogramas de audio, incluso series de tiempo (convolución 1D) |
| **Evítala cuando…** | Los datos son tabulares: el orden de las columnas no significa nada |

## 8. 🧠 Resumen en 5 puntos
1. Una convolución desliza un **filtro** pequeño por la imagen y, en cada posición, **multiplica y suma**: el resultado es un **mapa** de dónde aparece el patrón.
2. Distintos filtros detectan distintos patrones: bordes verticales, horizontales, desenfoque, nitidez…
3. En una CNN los filtros **no se diseñan: se aprenden** con descenso de gradiente.
4. Es **la misma lupa en toda la imagen** (pesos compartidos): pocos parámetros y detección en cualquier posición.
5. Perillas: **kernel** (qué tan grande es la lupa), **stride** (cuánto salta), **padding** (marco de ceros) y **número de filtros** (cuántos patrones busca). Tamaño de salida = (entrada + 2·padding − kernel) / stride + 1.

---
## 9. 🏋️ Ejercicios finales

### 9.1 Repaso rápido (opción múltiple)

**1.** Un filtro de 3 × 3 aplicado sobre una zona **completamente pareja** de la imagen con el detector `sobel_x` da…
- a) Un número muy grande · b) 0 · c) Un número negativo · d) Depende del brillo

**2.** Entrada de 64 × 64, kernel 5, stride 1, padding 2. La salida es de…
- a) 60 × 60 · b) 62 × 62 · c) 64 × 64 · d) 32 × 32

**3.** `nn.Conv2d(16, 32, kernel_size=3)` tiene…
- a) 32 × 3 × 3 + 32 = 320 parámetros
- b) 16 × 32 × 3 × 3 + 32 = 4,640 parámetros
- c) 16 × 3 × 3 = 144 parámetros
- d) Depende del tamaño de la imagen

**4.** ¿Qué hiperparámetro **no** cambia el número de parámetros de la capa?
- a) `kernel_size` · b) `out_channels` · c) `stride` · d) `in_channels`

**5.** ¿Por qué una capa convolucional sigue detectando un borde aunque se mueva de lugar?
- a) Porque usa el mismo filtro en todas las posiciones
- b) Porque tiene más parámetros que una capa densa
- c) Porque usa padding
- d) Porque los píxeles se dividen entre 255

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** Lo de la izquierda y lo de la derecha se cancelan.
2. **c)** (64 + 4 − 5)/1 + 1 = 64. Padding 2 con kernel 5 conserva el tamaño (`kernel_size // 2`).
3. **b)** Cada uno de los 32 filtros mira los 16 canales de entrada con un 3 × 3, más un sesgo por filtro.
4. **c)** El *stride* solo cambia cuántas posiciones se calculan, no el tamaño del filtro.
5. **a)** Pesos compartidos: la misma lupa recorre toda la imagen.
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** Una red convolucional entrenada para reconocer gatos detecta muy bien gatos **derechos**, pero falla con gatos **de cabeza**. ¿Por qué la convolución no resuelve sola este problema? ¿Qué harías? (Pista: Lección 9.8.)

**B. Caso 💼** Un hospital quiere detectar fracturas en radiografías de 2,000 × 2,000 píxeles en grises. Un ingeniero propone usar en la primera capa filtros de **51 × 51** "para ver bien los huesos". Calcula los parámetros de esa capa con 32 filtros y compárala con 32 filtros de 3 × 3. ¿Qué alternativa propondrías para "ver" zonas grandes sin filtros enormes?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** La convolución es la misma lupa en todas las **posiciones**, pero la lupa no gira: un filtro que aprendió "orejas puntiagudas hacia arriba" no se activa con orejas hacia abajo. La red solo reconoce lo que vio al entrenar. Solución: **data augmentation** con rotaciones (mostrarle gatos girados durante el entrenamiento), siempre que tenga sentido para el problema.

**B.** 32 × (1 × 51 × 51) + 32 = **83,264** parámetros contra 32 × (1 × 3 × 3) + 32 = **320**: unas 260 veces más, y muchísimo más cómputo en cada una de las ~4 millones de posiciones. Alternativa: **apilar** varias capas de 3 × 3 con reducciones de tamaño (*stride* o *pooling*): cada capa "ve" a través de la anterior, y así la red cubre zonas grandes con pocos parámetros. Es la idea de la **jerarquía de características** de la Lección 10.3 y de VGG (Lección 10.4).
</details>

### 9.3 🧩 Práctica integradora: "Diseña tus propios detectores"

Vas a trabajar con una imagen sintética que tiene un **cuadrado** (bordes horizontales y verticales) y una **línea diagonal**. Tu misión: diseñar filtros que encuentren cada cosa por separado.

**Paso 1 ▶️ — Mira la imagen**

In [ ]:
escena = np.zeros((40, 40))
escena[5:20, 5] = escena[5:20, 19] = 1          # lados verticales del cuadrado
escena[5, 5:20] = escena[19, 5:20] = 1          # lados horizontales del cuadrado
for i in range(15):
    escena[22 + i, 22 + i] = 1                  # diagonal ↘
plt.figure(figsize=(3.5, 3.5)); plt.imshow(escena, cmap="gray"); plt.axis("off"); plt.show()

**Paso 2 🔮 — Los detectores de siempre.** Antes de ejecutar: ¿qué partes de la escena encenderá `sobel_x`? ¿Y `sobel_y`? ¿Se encenderá la diagonal con alguno?

✍️ *Mi predicción:*

In [ ]:
def mostrar(filtros_dict):
    fig, ax = plt.subplots(1, len(filtros_dict), figsize=(4 * len(filtros_dict), 3.5))
    for a, (nombre, f) in zip(np.atleast_1d(ax), filtros_dict.items()):
        a.imshow(np.abs(convolucion(escena, f, padding=1)), cmap="magma"); a.set_title(nombre); a.axis("off")
    plt.show()

mostrar({"sobel_x (verticales)": sobel_x, "sobel_y (horizontales)": sobel_y})

<details><summary><b>👀 Ver respuesta</b></summary>

`sobel_x` enciende los **lados verticales** del cuadrado y `sobel_y` los **horizontales**. La diagonal se enciende **con los dos**, porque una línea diagonal tiene cambios de brillo tanto de izquierda a derecha como de arriba abajo. Así que ninguno de los dos la distingue de los bordes del cuadrado.
</details>

**Paso 3 ✏️ — Diseña un detector de diagonales.** Completa el filtro `mi_diagonal` (3 × 3) para que se encienda **mucho** sobre la diagonal ↘ y **poco** sobre los lados del cuadrado.

💡 Idea: ponle números **positivos** justo donde está la línea (la diagonal del filtro) y **negativos** en el resto, de modo que una zona pareja sume 0.

In [ ]:
mi_diagonal = np.array([[ 0,  0,  0],      # ✏️ diseña tu filtro aquí
                        [ 0,  1,  0],
                        [ 0,  0,  0]])

resp = np.abs(convolucion(escena, mi_diagonal, padding=1))
print(f"Respuesta máxima sobre la diagonal : {resp[22:37, 22:37].max():.1f}")
print(f"Respuesta máxima sobre el cuadrado : {resp[3:22, 3:22].max():.1f}")
mostrar({"mi_diagonal": mi_diagonal})

<details><summary><b>👀 Ver respuesta</b></summary>

Una solución:
```python
mi_diagonal = np.array([[ 2, -1, -1],
                        [-1,  2, -1],
                        [-1, -1,  2]])
```
Suma 0 (zona pareja → 0). Sobre la diagonal, los tres 2 caen sobre la línea → respuesta **6**. Sobre un lado recto del cuadrado, la línea cruza un 2 y dos −1 → respuesta **0**. En las esquinas del cuadrado queda algo de respuesta (hasta 2), pero mucho menos que en la diagonal.

El filtro que viene en la celda (solo un 1 en el centro) no detecta nada: copia la imagen tal cual (responde 1 en todas partes). Cópiale la solución y vuelve a ejecutar para ver la diferencia.
</details>

**Paso 4 ✋ — Del diseño a mano a una capa de verdad.** Vas a meter tus tres detectores (`sobel_x`, `sobel_y`, `mi_diagonal`) en **una sola** capa `nn.Conv2d` con 3 filtros. Antes de ejecutar responde:
1. ¿Qué valores pondrías en `in_channels` y `out_channels`?
2. ¿Qué forma tendrá la salida si la entrada es la escena de 40 × 40 y usas `padding=1`?
3. ¿Cuántos parámetros tiene la capa (sin sesgo)?

In [ ]:
SOLUCION_DIAGONAL = np.array([[2, -1, -1], [-1, 2, -1], [-1, -1, 2]])   # la de la respuesta del paso 3

tres_detectores = nn.Conv2d(in_channels=1, out_channels=3, kernel_size=3, padding=1, bias=False)
with torch.no_grad():
    for i, f in enumerate([sobel_x, sobel_y, SOLUCION_DIAGONAL]):
        tres_detectores.weight[i, 0] = torch.tensor(f, dtype=torch.float32)

mapas = tres_detectores(torch.tensor(escena, dtype=torch.float32)[None, None])
print("Forma de la salida:", tuple(mapas.shape))
print("Parámetros:", contar(tres_detectores))

fig, ax = plt.subplots(1, 3, figsize=(12, 3.5))
for a, m, t in zip(ax, mapas[0].detach().abs(), ["verticales", "horizontales", "diagonal"]):
    a.imshow(m, cmap="magma"); a.set_title(f"Mapa: {t}"); a.axis("off")
plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

1. `in_channels=1` (la escena está en grises) y `out_channels=3` (tres filtros).
2. `(1, 3, 40, 40)`: un lote de 1 imagen con **3 mapas de características** de 40 × 40 (el padding 1 conserva el tamaño).
3. 3 filtros × 1 canal × 3 × 3 = **27** parámetros.

¡Acabas de construir a mano la primera capa de una red convolucional! La única diferencia con una red de verdad es que allí esos 27 números **se aprenden** (Sección 3.2), y que hay muchos más filtros.
</details>

**Paso 5 🔮 — ¿Y si la escena se mueve?** Desplaza la escena 3 píxeles a la derecha y vuelve a pasarla por los tres detectores. ¿Cambian los valores máximos de cada mapa? ¿Y su posición?

In [ ]:
movida = np.roll(escena, 3, axis=1)
mapas_mov = tres_detectores(torch.tensor(movida, dtype=torch.float32)[None, None])[0].detach().abs()
for t, m0, m1 in zip(["verticales", "horizontales", "diagonal"], mapas[0].detach().abs().numpy(), mapas_mov.numpy()):
    c0 = np.unravel_index(m0.argmax(), m0.shape)[1]; c1 = np.unravel_index(m1.argmax(), m1.shape)[1]
    print(f"{t:<12} máximo original {m0.max():.0f} (columna {c0:>2}) · movida {m1.max():.0f} (columna {c1:>2})")

<details><summary><b>👀 Ver respuesta</b></summary>

Los **valores máximos son idénticos** y cada máximo aparece **3 columnas más a la derecha**: el mapa de características simplemente "se mueve" junto con la imagen. A esto se le llama **equivarianza a la traslación**: la convolución detecta el patrón esté donde esté, y su mapa dice dónde está.

Compáralo con el modelo "píxel por píxel" de la Lección 10.1, que bajaba de ≈87 % a ≈45 % de exactitud con solo 3 píxeles de desplazamiento. En la Lección 10.3 verás cómo el *pooling* convierte "el mapa se mueve" en "la respuesta final casi no cambia".
</details>

---
## 📚 Para profundizar
- 🎮 **Interactivo:** *Image Kernels Explained Visually* (setosa.io/ev/image-kernels): aplica filtros a una foto y modifica sus números en vivo.
- 🎮 **Interactivo:** *CNN Explainer* (poloclub.github.io/cnn-explainer): ve cada convolución de una red real paso a paso.
- 📘 **Lectura:** Dumoulin y Visin, *A guide to convolution arithmetic for deep learning* (inglés): animaciones de kernel, stride y padding.
- 📘 **Documentación:** PyTorch, `torch.nn.Conv2d`.

---
⬅️ **Anterior:** [10.1 Cómo "ve" una computadora una imagen](10.1-como-ve-una-computadora.ipynb) · ➡️ **Siguiente:** [10.3 Pooling y jerarquía de características](10.3-pooling-y-jerarquia.ipynb)